# AI product analytics workshop

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tbogoodnews/ai-product-usage-data-science-lab-students/blob/main/workshop.ipynb)

📘 **Techniques guide (PDF):** [one page per technique, with history, examples, and visuals](https://github.com/tbogoodnews/ai-product-usage-data-science-lab-students/blob/main/guide/techniques-guide.pdf)

Explore fictional AI product data with Python and pandas. Run the common setup, data tour, and join cells using the play button. Then choose a topic and run its examples and exercises in order; you can skip other topics. Replace each `pass` with your own code. If you get stuck, ask a partner or your instructor.

The data covers September through November 2026 in UTC. All accounts, behavior, hosting rates, and model properties are fictional. Prices are illustrative snapshots, not historical bills. See `schema.md` for details.

Start with the data tour and join (about 10 to 15 minutes). Then choose a topic: pricing, survival, trial cycling, activity, market baskets, switching, forecasting, enterprise, cache incidents, or cost concentration (about 15 to 25 minutes each). The go-further questions are optional.

In [ ]:
%matplotlib inline
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# DATA LOCATION: GitHub raw files. Leave this unchanged in Colab.
# An instructor can set WORKSHOP_BASE_URL to a local data folder for testing.
BASE_URL = os.environ.get(
    "WORKSHOP_BASE_URL",
    "https://raw.githubusercontent.com/tbogoodnews/ai-product-usage-data-science-lab-students/main/data/",
)

def load(name):
    return pd.read_csv(BASE_URL + name)

## Quick data tour

`users` has one row per account. `usage` has one row per account, model, route, host, price key and six-hour UTC bucket, plus other grouping fields. A user can appear many times. Tokens measure text processed, not people or messages. The raw tables remain available, but we will build a smaller table for most exercises.

In [ ]:
users = load("users.csv")
usage = load("usage.csv")
print("Accounts:", len(users), "| Usage rows:", len(usage))
display(users[["user_id", "region", "primary_provider", "business_id"]].head(3))
display(usage[["user_id", "time", "model_name", "model_call_count", "aic"]].head(3))

### Quick pandas examples

`head()` shows a few rows. Select columns with brackets. `groupby` makes groups; `sum()` adds values, while `nunique()` counts different values. `sort_values()` puts the largest first. Run this example, then change a column name to explore.

In [ ]:
display(usage[["provider", "user_id", "model_call_count"]].head())
calls_by_provider = usage.groupby("provider")["model_call_count"].sum()
display(calls_by_provider.sort_values(ascending=False))
display(usage.groupby("provider")["user_id"].nunique())

## 1. Data engineering: join the tables

Select a small set of useful columns, then join usage to accounts for region and declared provider. A left join keeps every usage row. `validate` checks that the account table has only one row per user, so the join cannot silently multiply usage. `analysis` retains the original six-hour usage-aggregate grain, not one row per user or request. See the techniques guide for a field dictionary.

In [ ]:
# Keep raw usage available, but work with a smaller set of columns.
analysis_columns = ["user_id", "time", "user_type", "model_name", "provider",
                    "model_call_count", "user_request_count", "total_input_tokens",
                    "aic", "hosting_cost_usd", "business_id"]
analysis = usage[analysis_columns].merge(
    users[["user_id", "region", "primary_provider"]],
    on="user_id", how="left", validate="many_to_one", indicator=True
)
print("Rows before:", len(usage), "| Rows after:", len(analysis))
print(analysis["_merge"].value_counts())
analysis = analysis.drop(columns="_merge")
display(analysis.head())

**Try 1:** Count usage rows by region. Sort from most to fewest.

In [ ]:
# TODO: write your code here
pass

**Interpret:** Write one sentence about your result.

**Try 2:** Count distinct accounts in each region. Hint: use `nunique()`.

In [ ]:
# TODO: write your code here
pass

**Interpret:** Write one sentence about your result.

**Try 3:** For each region, calculate total model calls per distinct account. Divide the sums, not individual row ratios.

In [ ]:
# TODO: write your code here
pass

**Interpret:** Write one sentence about your result.

**Go further:** How would the result change if you included accounts with no usage?

## 2. Pricing and a margin-like comparison

A **turn** means one user request. A request can cause several model calls, including agent steps, retries, or fallbacks. For this made-up old scheme we charge **once per admitted request**, never once per model call; blocked attempts are not billed. The hypothetical AIC scheme charges for every AIC used. The editable prices are teaching assumptions, not reconstructed historical terms. Neither scheme includes subscriptions, allowances, overage caps, taxes, or other costs.

The dataset's `aic_usd_proxy` is AIC multiplied by $0.01, **not revenue**. Our contribution and margin apply only to these simplified scenarios, not actual profit or margin. API list price is a separate benchmark.

Prices are per million tokens. Cache writes are already part of uncached tokens, so use `ordinary_billed_input_tokens` for ordinary input and count writes only once. Missing write rates mean writes are unsupported.

In [ ]:
api = load("api_prices.csv")
copilot = load("copilot_aic_prices.csv")
hosts = load("hosting_rates.csv")
# Pricing needs a few extra raw columns that are not in the compact analysis table.
pricing_columns = ["model_name", "host", "api_price_key", "copilot_price_key",
                   "ordinary_billed_input_tokens", "cache_read_tokens", "cache_write_tokens",
                   "output_tokens", "user_request_count", "model_call_count",
                   "api_list_usd", "hosting_cost_usd", "aic"]
priced = usage[pricing_columns].merge(
    api[["price_key", "input_rate", "cache_read_rate", "cache_write_rate", "output_rate"]],
    left_on="api_price_key", right_on="price_key", validate="many_to_one"
).merge(hosts[["host", "multiplier"]], on="host", validate="many_to_one")
priced["recalculated_api_usd"] = (
    priced["ordinary_billed_input_tokens"] * priced["input_rate"]
    + priced["cache_read_tokens"] * priced["cache_read_rate"]
    + priced["cache_write_tokens"] * priced["cache_write_rate"].fillna(0)
    + priced["output_tokens"] * priced["output_rate"]
) / 1_000_000
print("API total, stored:", round(priced["api_list_usd"].sum(), 2))
print("API total, recalculated:", round(priced["recalculated_api_usd"].sum(), 2))

requests = load("user_requests.csv")[["user_request_id", "status"]]
calls = load("model_requests.csv")[["user_request_id", "request_id", "call_role"]]
admitted = requests[requests["status"] == "admitted"]
calls_per_turn = calls.groupby("user_request_id")["request_id"].nunique()
example_id = calls_per_turn.idxmax()
# Editable teaching assumptions (USD). These are not real prices.
old_usd_per_turn = 0.04
new_usd_per_aic = 0.01
print("Admitted turns:", len(admitted), "| Blocked attempts:", (requests["status"] == "blocked").sum())
print("Model calls:", len(calls), "| Turns with multiple calls:", (calls_per_turn > 1).sum())
print(f"Example {example_id}: {calls_per_turn.loc[example_id]} calls, one admitted turn, old charge ${old_usd_per_turn:.2f}")
print("Calls by role:")
display(calls["call_role"].value_counts())
# user_request_count assigns each admitted turn once to its initial route's aggregate.
assert priced["user_request_count"].sum() == len(admitted)
priced["new_bill_usd"] = priced["aic"] * new_usd_per_aic

**Try 1:** Calculate the old modeled customer bill using admitted turns, not model calls. Compare it with the hypothetical AIC bill, API list cost and provider hosting cost.

In [ ]:
# TODO: write your code here
pass

**Interpret:** Write one sentence about your result.

**Try 2:** Group modeled bills and hosting costs by model. For each scheme, subtract hosting cost from the bill to get modeled contribution. The old turn charge is assigned to the initial route, not every model called.

In [ ]:
# TODO: write your code here
pass

**Interpret:** Write one sentence about your result.

**Try 3:** For each scheme, calculate overall modeled contribution margin: `(total bill - total hosting cost) / total bill`. Change the assumptions and compare. Why is this not actual margin?

In [ ]:
# TODO: write your code here
pass

**Interpret:** Write one sentence about your result.

**Go further:** Join `copilot` on `copilot_price_key` and check how its token rates differ from API rates. How might a price change affect your comparison?

## 3. Survival: how long does a subscription spell last?

A subscription row is a SKU interval, not necessarily a customer leaving. A switch to another SKU ends a spell too. Intervals include the start and exclude the end: `[start, end)`.

We study only spells that **start within** our observation window. That avoids treating older spells as new. A spell still running on December 1 is right-censored: we know it lasted at least that long, but not when it ends. This is not account-closure survival.

In [ ]:
subscriptions = load("subscriptions.csv")
start = pd.Timestamp("2026-09-01", tz="UTC")
end = pd.Timestamp("2026-12-01", tz="UTC")
subscriptions["start"] = pd.to_datetime(subscriptions["valid_from"], utc=True)
subscriptions["end"] = pd.to_datetime(subscriptions["valid_to"], utc=True)
spells = subscriptions[(subscriptions["start"] >= start) & (subscriptions["start"] < end)].copy()
spells["observed_end"] = spells["end"].clip(upper=end)
spells["days"] = (spells["observed_end"] - spells["start"]).dt.total_seconds() / 86400
spells["ended"] = spells["end"] < end
print("New spells:", len(spells), "| Right-censored:", (~spells["ended"]).sum())
display(spells[["sku", "days", "ended"]].head())

**Try 1:** Count new spells and ended spells for each SKU.

In [ ]:
# TODO: write your code here
pass

**Interpret:** Write one sentence about your result.

**Try 2:** Find the median observed days for each SKU. Why is this not necessarily the median complete subscription length?

In [ ]:
# TODO: write your code here
pass

**Interpret:** Write one sentence about your result.

**Try 3:** Build a Kaplan-Meier curve for all new spells: at each observed duration, count endings and all spells still at risk, then multiply `(1 - endings / at_risk)` across times. Plot the result. Hint: sort by `days`; censored spells leave the risk set after their observed time.

In [ ]:
# TODO: write your code here
pass

**Interpret:** Write one sentence about your result.

**Go further:** Make separate curves by SKU. How does the short observation window limit comparisons?

## 4. Trial cycling: investigate shared computers

Multiple trial accounts on a computer can be a useful signal to investigate, **not proof of fraud**. A family, school, or shared machine can explain it. Count distinct accounts, not device rows. Only count a mapping when its time interval overlaps that account's trial interval.

In [ ]:
devices = load("device_mappings.csv")
trials = load("subscriptions.csv")
trials = trials[trials["sku"] == "pro_trial"].copy()
trials["start"] = pd.to_datetime(trials["valid_from"], utc=True)
trials["end"] = pd.to_datetime(trials["valid_to"], utc=True)
devices["map_start"] = pd.to_datetime(devices["valid_from"], utc=True)
devices["map_end"] = pd.to_datetime(devices["valid_to"], utc=True)
trial_devices = devices.merge(trials[["user_id", "start", "end"]], on="user_id", how="inner")
trial_devices = trial_devices[
    (trial_devices["map_start"] < trial_devices["end"]) &
    (trial_devices["start"] < trial_devices["map_end"])
]
print("Overlapping trial-device mappings:", len(trial_devices))
display(trial_devices[["computer_id", "user_id", "map_start", "start"]].head())

**Try 1:** Count distinct trial accounts per computer and show the largest counts.

In [ ]:
# TODO: write your code here
pass

**Interpret:** Write one sentence about your result.

**Try 2:** How many computers have at least two trial accounts? What fraction of trial-associated computers is that?

In [ ]:
# TODO: write your code here
pass

**Interpret:** Write one sentence about your result.

**Try 3:** Inspect the mapping and trial dates for one shared computer. Do the trial periods overlap or follow one another?

In [ ]:
# TODO: write your code here
pass

**Interpret:** Write one sentence about your result.

**Go further:** Design a second, privacy-conscious signal. What innocent explanations and false positives would you check?

## 5. Activity: DAU, WAU, WEU, and WDU

An active account has at least one model call on a UTC calendar day. DAU counts distinct active accounts that day. For each day, WAU, WEU, and WDU count accounts active on at least 1, 2, and 5 days, respectively, in the seven days ending that day. A user with many calls on one day still has only one active day.

The graph starts on September 7, when a full seven-day window is available. These are account counts, not counts of unique people. The data ends on November 30.

In [ ]:
import numpy as np

# Turn each six-hour bucket into one UTC day and remove duplicate account-days.
activity = usage.loc[usage["model_call_count"] > 0, ["user_id", "time", "model_name", "user_type", "total_input_tokens", "output_tokens", "model_call_count"]].copy()
activity["day"] = pd.to_datetime(activity["time"], utc=True).dt.floor("D")
activity["tokens"] = activity["total_input_tokens"] + activity["output_tokens"]
account_days = activity[["user_id", "day"]].drop_duplicates()
days = pd.date_range("2026-09-01", "2026-11-30", freq="D", tz="UTC")

daily = account_days.groupby("day")["user_id"].nunique().reindex(days, fill_value=0)
records = []
for day in days[6:]:
    week = account_days[(account_days["day"] > day - pd.Timedelta(days=7)) & (account_days["day"] <= day)]
    active_days = week.groupby("user_id")["day"].nunique()
    wau = (active_days >= 1).sum()
    weu = (active_days >= 2).sum()
    wdu = (active_days >= 5).sum()
    records.append({"day": day, "DAU": daily.loc[day], "WAU": wau, "WEU": weu, "WDU": wdu})
metrics = pd.DataFrame(records).set_index("day")
metrics["WEU / WAU"] = metrics["WEU"] / metrics["WAU"]
metrics["WDU / WAU"] = metrics["WDU"] / metrics["WAU"]
display(metrics.tail())
metrics[["DAU", "WAU", "WEU", "WDU"]].plot(figsize=(10, 4), ylabel="Accounts", title="Daily and trailing seven-day activity")
plt.tight_layout()
plt.show()
metrics[["WEU / WAU", "WDU / WAU"]].plot(figsize=(10, 3), ylim=(0, 1), ylabel="Share of WAU", title="Activity intensity")
plt.tight_layout()
plt.show()

A product team can track the ratios to see whether weekly active accounts return on more days. A change is a signal to investigate, not proof that a feature caused it. These measures do not include inactive accounts in the weekly denominator.

For the next questions, use the final seven-day window, November 24 to 30. Each account is assigned its most recently observed SKU in that window. For model category, assign its most-used provider by tokens in the window. Ties are settled alphabetically. Accounts can change SKU, so this is a snapshot, not a complete subscription history.

In [ ]:
snapshot_day = pd.Timestamp("2026-11-30", tz="UTC")
snapshot = activity[(activity["day"] > snapshot_day - pd.Timedelta(days=7)) & (activity["day"] <= snapshot_day)].copy()
snapshot_days = snapshot.groupby("user_id")["day"].nunique().rename("active_days")
# Latest observed SKU within the window; same-time ties use alphabetical order.
snapshot_sku = (snapshot.sort_values(["user_id", "time", "user_type"])
                .drop_duplicates("user_id", keep="last")[["user_id", "user_type"]])
catalog = load("model_catalog.csv")
snapshot = snapshot.merge(
    catalog[["model_name", "provider"]], on="model_name", validate="many_to_one")
provider_tokens = snapshot.groupby(["user_id", "provider"], as_index=False)["tokens"].sum()
top_provider = (provider_tokens.sort_values(["user_id", "tokens", "provider"], ascending=[True, False, True])
                .drop_duplicates("user_id")[["user_id", "provider"]])
snapshot_users = (snapshot_days.to_frame().reset_index()
                  .merge(snapshot_sku, on="user_id", validate="one_to_one")
                  .merge(top_provider, on="user_id", validate="one_to_one"))
snapshot_users["wdu"] = snapshot_users["active_days"] >= 5

**Try 5.1:** Compare WDU / WAU for each SKU. Which observed SKU has the highest share? Show the number of weekly active accounts too.

In [ ]:
# TODO: write your code here
pass

**Try 5.2:** Compare WDU / WAU by the provider of each account's most-used model. Which provider group has the highest share?

In [ ]:
# TODO: write your code here
pass

## 6. Market baskets: two weeks

A basket is the set of models an account used. Compare the week before the fictional Claude Opus 5 launch with its launch week. We count accounts with calls in each week; the two groups need not contain the same accounts. A routed model is not always an explicit choice.

In [ ]:
week_a = pd.Timestamp("2026-09-15", tz="UTC")
week_b = pd.Timestamp("2026-09-22", tz="UTC")
week_length = pd.Timedelta(days=7)
model_usage = usage.loc[usage["model_call_count"] > 0, ["user_id", "time", "model_name", "provider", "total_input_tokens", "output_tokens", "model_call_count"]].copy()
model_usage["day"] = pd.to_datetime(model_usage["time"], utc=True).dt.floor("D")
model_usage["tokens"] = model_usage["total_input_tokens"] + model_usage["output_tokens"]
model_usage["week"] = np.where(
    (model_usage["day"] >= week_a) & (model_usage["day"] < week_a + week_length), "A",
    np.where((model_usage["day"] >= week_b) & (model_usage["day"] < week_b + week_length), "B", "outside"))
two_weeks = model_usage[model_usage["week"] != "outside"].copy()
baskets = two_weeks.groupby(["week", "user_id"])["model_name"].agg(lambda x: sorted(set(x)))
display(baskets.head())
print("Week A: September 15-21 | Week B: September 22-28")

**Try 6.1:** Compare the mean and median number of models per active account in the two weeks. Is there a shift to more models?

In [ ]:
# TODO: write your code here
pass

**Try 6.2:** For each account and week, find the provider with the largest share of its tokens. Compare the share of accounts with at least 80% of tokens from one provider. Does this show loyalty?

In [ ]:
# TODO: write your code here
pass

## 7. Switching: most-used model before and after a launch

This example compares October 20-26 with October 27-November 2, around the fictional Claude Opus 5.5 launch. We select each account's model with the most input plus output tokens in each week. We include only accounts active in both weeks; ties are broken by model name. A switch in this measure is not proof of an intentional choice, and simultaneous launches or routing may also matter.

The Sankey shows flows between models. The six models with the most tokens across both weeks are shown by name; the rest are grouped as Other in the picture only. Calculations below use the original model names. A static picture follows for viewers that cannot display the interactive chart.

In [ ]:
import plotly.graph_objects as go
from matplotlib.path import Path
from matplotlib.patches import PathPatch, Rectangle

switch_a = pd.Timestamp("2026-10-20", tz="UTC")
switch_b = pd.Timestamp("2026-10-27", tz="UTC")
switch_usage = usage.loc[usage["model_call_count"] > 0, ["user_id", "time", "user_type", "model_name", "total_input_tokens", "output_tokens", "model_call_count"]].copy()
switch_usage["day"] = pd.to_datetime(switch_usage["time"], utc=True).dt.floor("D")
switch_usage["tokens"] = switch_usage["total_input_tokens"] + switch_usage["output_tokens"]
switch_usage["week"] = np.where(
    (switch_usage["day"] >= switch_a) & (switch_usage["day"] < switch_a + pd.Timedelta(days=7)), "A",
    np.where((switch_usage["day"] >= switch_b) & (switch_usage["day"] < switch_b + pd.Timedelta(days=7)), "B", "outside"))
switch_usage = switch_usage[switch_usage["week"] != "outside"]
model_totals = switch_usage.groupby(["week", "user_id", "model_name"], as_index=False)["tokens"].sum()
top_models = (model_totals.sort_values(["week", "user_id", "tokens", "model_name"], ascending=[True, True, False, True])
              .drop_duplicates(["week", "user_id"]))
paired = top_models.pivot(index="user_id", columns="week", values="model_name").dropna().rename(columns={"A": "before", "B": "after"})
paired["switched"] = paired["before"] != paired["after"]
catalog = load("model_catalog.csv")
provider_map = catalog.set_index("model_name")["provider"]
paired["before_provider"] = paired["before"].map(provider_map)
paired["after_provider"] = paired["after"].map(provider_map)
last_a = (switch_usage[switch_usage["week"] == "A"].sort_values(["user_id", "time", "user_type"])
          .drop_duplicates("user_id", keep="last").set_index("user_id")["user_type"])
paired["sku_before"] = paired.index.map(last_a)
business_map = users.set_index("user_id")["business_id"]
paired["account_type"] = np.where(paired.index.map(business_map).notna(), "Business", "Individual")
print("Accounts active in both weeks:", len(paired))
print(f"Most-used model changed: {paired['switched'].mean():.1%}")
new_model = "Claude Opus 5.5"
print("Accounts whose top model became", new_model + ":", (paired["after"] == new_model).sum())

shown = model_totals.groupby("model_name")["tokens"].sum().nlargest(6).index
plot_pairs = paired.assign(
    source=paired["before"].where(paired["before"].isin(shown), "Other"),
    target=paired["after"].where(paired["after"].isin(shown), "Other"))
flows = plot_pairs.groupby(["source", "target"]).size().reset_index(name="accounts")
names = sorted(set(flows["source"]) | set(flows["target"]))
left = ["Before: " + name for name in names]
right = ["After: " + name for name in names]
source_indices = flows["source"].map({name: i for i, name in enumerate(names)})
target_indices = flows["target"].map({name: i + len(names) for i, name in enumerate(names)})
fig = go.Figure(go.Sankey(
    node=dict(label=left + right, pad=15, thickness=16),
    link=dict(source=source_indices, target=target_indices, value=flows["accounts"])))
fig.update_layout(title="Most-used model: week before to launch week", font_size=10, height=550)
fig.show(renderer="plotly_mimetype")

# Static fallback: line width shows the number of accounts in each flow.
fig, ax = plt.subplots(figsize=(12, 6))
positions = {name: 1 - (i + 0.5) / len(names) for i, name in enumerate(names)}
for _, row in flows.iterrows():
    y1, y2 = positions[row["source"]], positions[row["target"]]
    path = Path([(0.12, y1), (0.4, y1), (0.6, y2), (0.88, y2)], [Path.MOVETO, Path.CURVE4, Path.CURVE4, Path.CURVE4])
    ax.add_patch(PathPatch(path, facecolor="none", edgecolor="steelblue", alpha=0.25, lw=max(0.5, row["accounts"] / 12)))
for name, y in positions.items():
    ax.add_patch(Rectangle((0.105, y - 0.02), 0.015, 0.04, color="navy"))
    ax.add_patch(Rectangle((0.88, y - 0.02), 0.015, 0.04, color="navy"))
    ax.text(0.1, y, name, ha="right", va="center", fontsize=8)
    ax.text(0.9, y, name, ha="left", va="center", fontsize=8)
ax.set(xlim=(-0.18, 1.18), ylim=(0, 1), title="Static flow view: wider means more accounts")
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
provider_switch = paired.groupby("before_provider").agg(accounts=("switched", "size"), switched=("switched", "sum"))
provider_switch["switch_share"] = provider_switch["switched"] / provider_switch["accounts"]
display(provider_switch.sort_values("switch_share", ascending=False))
sku_switch = paired.groupby("sku_before").agg(accounts=("switched", "size"), switched=("switched", "sum"))
sku_switch["switch_share"] = sku_switch["switched"] / sku_switch["accounts"]
display(sku_switch.sort_values("switch_share", ascending=False))
print(f"Among accounts active in both weeks, {paired['switched'].mean():.1%} changed top model; {int((paired['after'] == new_model).sum())} had {new_model} as their new top model.")
high = provider_switch["switch_share"].idxmax()
high_sku = sku_switch["switch_share"].idxmax()
print(f"The highest observed shares were {high} ({provider_switch.loc[high, 'switch_share']:.1%}) by starting provider and {high_sku} ({sku_switch.loc[high_sku, 'switch_share']:.1%}) by SKU. The launch did not necessarily cause these switches.")

**Try 7.1:** Compare the share whose top model changed to a different provider, by their latest observed SKU in week A. Show the number of accounts in each group.

In [ ]:
# TODO: write your code here
pass

**Try 7.2:** Compare switching between business and individual accounts. What do you notice?

In [ ]:
# TODO: write your code here
pass

## 8. Bayesian launch forecasting: a worked example

This section is a demonstration. You do not need to fill in any code. We use a previous fictional model launch to make a starting guess for the new model. A Bayesian update combines that starting guess, called a prior, with a few new observations. The result is called a posterior.

We compare Claude Opus 5 (September 22) with Claude Opus 5.5 (October 27). We divide each day's model tokens by all accounts active on that day, including accounts that did not use that model. We use the first three days of the new launch to update the guess, then hold the average number of active accounts in those three days fixed for the forecast.

In [ ]:
from scipy.stats import norm

# Tokens here include input and output. Active accounts include users of any model.
launch_usage = usage.loc[usage["model_call_count"] > 0, ["user_id", "time", "model_name", "total_input_tokens", "output_tokens", "model_call_count"]].copy()
launch_usage["day"] = pd.to_datetime(launch_usage["time"], utc=True).dt.floor("D")
launch_usage["tokens"] = launch_usage["total_input_tokens"] + launch_usage["output_tokens"]
active_per_day = launch_usage.groupby("day")["user_id"].nunique()

def launch_daily(model, launch_date, count=7):
    dates = pd.date_range(launch_date, periods=count, freq="D", tz="UTC")
    tokens = launch_usage[launch_usage["model_name"] == model].groupby("day")["tokens"].sum().reindex(dates, fill_value=0)
    active = active_per_day.reindex(dates)
    return pd.DataFrame({"tokens": tokens.to_numpy(), "active_accounts": active.to_numpy(),
                         "per_active": (tokens / active).to_numpy()},
                        index=pd.Index(range(count), name="days_since_launch"))

previous = launch_daily("Claude Opus 5", "2026-09-22")
new = launch_daily("Claude Opus 5.5", "2026-10-27")
first_n = 3
prior_mean = previous["per_active"].mean()
# Use the older launch's day-to-day variation as an illustrative uncertainty.
day_sd = max(previous["per_active"].std(), 1.0)
prior_sd = day_sd
observed = new["per_active"].iloc[:first_n]
# Normal prior plus a normal observation model gives a normal posterior.
posterior_variance = 1 / (1 / prior_sd**2 + first_n / day_sd**2)
posterior_mean = posterior_variance * (prior_mean / prior_sd**2 + observed.sum() / day_sd**2)
posterior_sd = np.sqrt(posterior_variance)
lower, upper = norm.ppf([0.025, 0.975], loc=posterior_mean, scale=posterior_sd)
assumed_active = new["active_accounts"].iloc[:first_n].mean()
forecast_mean = max(0, posterior_mean) * assumed_active
forecast_low = max(0, lower) * assumed_active
forecast_high = max(0, upper) * assumed_active
print(f"Prior mean: {prior_mean:,.0f} tokens per active account per day")
print(f"Posterior mean after {first_n} days: {posterior_mean:,.0f} tokens per active account per day")
print(f"Assumed daily active accounts: {assumed_active:,.0f}")
display(new.iloc[:first_n])

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(7)
ax.plot(x[:first_n], new["tokens"].iloc[:first_n], "o-", label="New launch observed")
ax.plot(x, previous["tokens"], "o--", alpha=0.6, label="Previous launch observed")
ax.plot(x[first_n:], np.repeat(forecast_mean, 7 - first_n), "o-", label="Forecast mean")
ax.fill_between(x[first_n:], forecast_low, forecast_high, alpha=0.2, label="95% interval for mean")
ax.axvline(first_n - 0.5, color="grey", linestyle=":")
ax.set(xlabel="Days since launch (day 0 is launch day)", ylabel="Tokens per day", title="Launch forecast after the first three days")
ax.legend()
plt.tight_layout()
plt.show()

**What to notice**
- The prior comes from one older launch. It is a starting guess, not a fact.
- The new model's first three days move that guess. Days 3-6 are not used in the update.
- The shaded interval describes uncertainty about the average, not the range of individual daily outcomes. It assumes the simple model and fixed active-account count are reasonable.
- This is a classroom example, not a reliable business forecast. Launches, adoption, traffic, and daily variation can differ.

Try changing `first_n` from 3 to 2 or 4 and compare the update. Do not use days you are trying to forecast as inputs.

## 9. Enterprise analysis

Business accounts have a `business_id`. The business table describes a regional working schedule. Compare businesses without assuming that more calls mean better productivity. The counts below are model calls, not people or user actions.

In [ ]:
businesses = load("businesses.csv")
enterprise = analysis[analysis["business_id"].notna()].merge(
    businesses[["business_id", "region", "timezone"]],
    on="business_id", how="left", validate="many_to_one", suffixes=("_user", "_business")
)
print("Business usage rows:", len(enterprise), "| Businesses:", enterprise["business_id"].nunique())
display(enterprise[["business_id", "region_business", "model_name", "model_call_count"]].head())

**Try 1:** For each business, count all eligible accounts and total calls. Start from `users` to include accounts with no usage.

In [ ]:
# TODO: write your code here
pass

**Interpret:** Write one sentence about your result.

**Try 2:** Calculate calls per eligible account for each business. Use total calls divided by all business accounts, including those with no usage.

In [ ]:
# TODO: write your code here
pass

**Interpret:** Write one sentence about your result.

**Try 3:** Find the most-used model by total calls for each business. Hint: group by both business and model first.

In [ ]:
# TODO: write your code here
pass

**Interpret:** Write one sentence about your result.

**Go further:** Using business time zones and schedules, compare usage during and outside working hours. Usage is stored in six-hour UTC buckets, so exact working-hour attribution is approximate.

## 10. Detecting cache-rate drops

A cache read reuses input already stored instead of charging the ordinary input rate. A lower cache-read rate can raise modeled hosting cost. We look at each provider's six-hour UTC buckets: add cache-read tokens and divide by all input tokens. This is a token-weighted rate, not an average of row percentages.

For each bucket, compare the rate with the previous 28 six-hour buckets (seven days), excluding the bucket being tested. We require at least eight earlier buckets with input and 100,000 current input tokens. Flag a drop of at least two percentage points. These are adjustable teaching rules, not proof of an incident. Changing model mix or random variation can cause false alarms, and a short incident can be diluted in a six-hour bucket.

In [ ]:
# Select only fields needed for cache rates and the cost estimate.
cache_columns = ["time", "provider", "host", "api_price_key", "total_input_tokens",
                 "ordinary_billed_input_tokens", "cache_read_tokens", "cache_write_tokens",
                 "output_tokens", "hosting_cost_usd"]
cache = usage[cache_columns].copy()
cache["time"] = pd.to_datetime(cache["time"], utc=True)
bucket = cache.groupby(["provider", "time"]).agg(
    input_tokens=("total_input_tokens", "sum"), read_tokens=("cache_read_tokens", "sum"),
    ordinary_tokens=("ordinary_billed_input_tokens", "sum"), observed_hosting_usd=("hosting_cost_usd", "sum"))
# Include empty time buckets so 28 buckets really means seven calendar days.
all_times = pd.date_range("2026-09-01", "2026-11-30 18:00", freq="6h", tz="UTC")
grid = pd.MultiIndex.from_product([sorted(cache["provider"].unique()), all_times], names=["provider", "time"])
bucket = bucket.reindex(grid, fill_value=0).reset_index()
bucket["read_rate"] = bucket["read_tokens"] / bucket["input_tokens"].replace(0, np.nan)
bucket["has_input"] = (bucket["input_tokens"] > 0).astype(int)
grouped = bucket.groupby("provider")
prior_read = grouped["read_tokens"].transform(lambda x: x.shift().rolling(28, min_periods=1).sum())
prior_input = grouped["input_tokens"].transform(lambda x: x.shift().rolling(28, min_periods=1).sum())
prior_buckets = grouped["has_input"].transform(lambda x: x.shift().rolling(28, min_periods=1).sum())
bucket["baseline_rate"] = (prior_read / prior_input.replace(0, np.nan)).where(prior_buckets >= 8)
bucket["drop"] = bucket["baseline_rate"] - bucket["read_rate"]
bucket["flag"] = (bucket["input_tokens"] >= 100_000) & (bucket["drop"] >= 0.02)
print("Flagged provider-buckets:", int(bucket["flag"].sum()))
display(bucket.loc[bucket["flag"], ["provider", "time", "input_tokens", "read_rate", "baseline_rate", "drop"]])

### Estimating the avoidable cost

This is a **what-if**, not an observed loss. In flagged buckets, imagine the read rate returned to the prior baseline. Convert only enough ordinary billed input tokens to reads to reach that baseline, capped at the ordinary input available. Keep total input, cache-write tokens, output tokens, model prices and hosting multipliers unchanged. The savings use each row's `(ordinary input rate - cache read rate)` and its host multiplier. Cache writes and outputs are not converted or counted again. Baseline choice and model mix make this estimate uncertain.

In [ ]:
cache_rates = cache.merge(
    load("api_prices.csv")[["price_key", "input_rate", "cache_read_rate"]],
    left_on="api_price_key", right_on="price_key", validate="many_to_one"
).merge(load("hosting_rates.csv")[["host", "multiplier"]], on="host", validate="many_to_one")
flagged = bucket[bucket["flag"]].copy()
flagged["extra_reads"] = (flagged["baseline_rate"] * flagged["input_tokens"] - flagged["read_tokens"]).clip(lower=0)
flagged["extra_reads"] = flagged[["extra_reads", "ordinary_tokens"]].min(axis=1)
cost_rows = cache_rates.merge(
    flagged[["provider", "time", "extra_reads", "ordinary_tokens"]],
    on=["provider", "time"], how="inner", validate="many_to_one")
# Spread the extra reads across the ordinary input in that provider-bucket.
cost_rows["shifted_tokens"] = (cost_rows["extra_reads"] * cost_rows["ordinary_billed_input_tokens"]
                                / cost_rows["ordinary_tokens"].replace(0, np.nan)).fillna(0)
cost_rows["estimated_saving_usd"] = (
    cost_rows["shifted_tokens"] * (cost_rows["input_rate"] - cost_rows["cache_read_rate"])
    * cost_rows["multiplier"] / 1_000_000)
incident_cost = cost_rows.groupby(["provider", "time"]).agg(
    observed_hosting_usd=("hosting_cost_usd", "sum"), estimated_saving_usd=("estimated_saving_usd", "sum"))
display(incident_cost)
print("Estimated saving across flagged buckets: $", round(incident_cost["estimated_saving_usd"].sum(), 2))

**Try 10.1:** For each provider, count flagged buckets and add its estimated savings. Which provider has the largest estimated saving?

In [ ]:
# TODO: write your code here
pass

**Go further:** Change the drop threshold and minimum input volume. How do the number of flags and false-alarm risk change?

## 11. Econometrics: who accounts for the cost?

We study **modeled hosting cost**, not customer revenue, for the `pro` SKU in October 2026 UTC. Include every account with a pro subscription overlapping the month, even if it has no pro usage; those accounts have zero cost. For accounts that change SKU, include only their usage rows labeled `pro` in October. This is cost during the observed pro period, not the account's total cost across all SKUs.

A Lorenz curve sorts accounts from lowest to highest cost and plots cumulative share of cost against cumulative share of accounts. The diagonal is perfect equality. A separate chart shows cost per account in dollars against its cumulative account percentile. This is descriptive concentration, not a causal explanation.

In [ ]:
month_start = pd.Timestamp("2026-10-01", tz="UTC")
month_end = pd.Timestamp("2026-11-01", tz="UTC")
chosen_sku = "pro"
subscriptions_cost = load("subscriptions.csv")[["user_id", "sku", "valid_from", "valid_to"]].copy()
subscriptions_cost["valid_from"] = pd.to_datetime(subscriptions_cost["valid_from"], utc=True)
subscriptions_cost["valid_to"] = pd.to_datetime(subscriptions_cost["valid_to"], utc=True)
eligible = subscriptions_cost[
    (subscriptions_cost["sku"] == chosen_sku) &
    (subscriptions_cost["valid_from"] < month_end) &
    (subscriptions_cost["valid_to"] > month_start)
][["user_id"]].drop_duplicates()
cost_usage = usage[["user_id", "time", "user_type", "hosting_cost_usd"]].copy()
cost_usage["time"] = pd.to_datetime(cost_usage["time"], utc=True)
monthly = cost_usage[(cost_usage["time"] >= month_start) & (cost_usage["time"] < month_end)
                     & (cost_usage["user_type"] == chosen_sku)]
cost_by_user = monthly.groupby("user_id")["hosting_cost_usd"].sum().rename("cost_usd")
cost_accounts = eligible.merge(cost_by_user, on="user_id", how="left", validate="one_to_one").fillna({"cost_usd": 0})
cost_accounts = cost_accounts.sort_values(["cost_usd", "user_id"]).reset_index(drop=True)
cost_accounts["user_share"] = (np.arange(len(cost_accounts)) + 1) / len(cost_accounts)
cost_accounts["cost_share"] = cost_accounts["cost_usd"].cumsum() / cost_accounts["cost_usd"].sum()
print("Eligible pro accounts:", len(cost_accounts), "| Zero-cost accounts:", int((cost_accounts["cost_usd"] == 0).sum()))
display(cost_accounts.head())
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(np.r_[0, cost_accounts["user_share"]], np.r_[0, cost_accounts["cost_share"]], label="Hosting cost")
axes[0].plot([0, 1], [0, 1], "--", label="Equal cost")
axes[0].set(xlabel="Cumulative share of accounts", ylabel="Cumulative share of cost", title="Lorenz curve: October pro")
axes[0].legend()
axes[1].plot(cost_accounts["user_share"] * 100, cost_accounts["cost_usd"])
axes[1].set(xlabel="Cumulative account percentile", ylabel="Hosting cost per account (USD)", title="Cost per account")
plt.tight_layout()
plt.show()

**Try 11.1:** What share of modeled hosting cost comes from the highest-cost 20% of eligible pro accounts? Count whole accounts and report the actual number selected.

In [ ]:
# TODO: write your code here
pass

**Go further:** Repeat for another SKU or month. Explain how including zero-cost accounts changes the curve.